# Human-in-the-Loop Approval for Irreversible Agent Actions

Anthropic's [autonomy research](https://www.anthropic.com/news/measuring-agent-autonomy) — ~1M production tool calls — found **0.8% of agent actions are irreversible** (moving money, deleting data, external communication), and that these should *"require mandatory human approval before execution."*

**The one lesson this recipe exists to teach:** approval is **not** another tool the model is instructed to call. It is an **executor-side precondition** — a state transition that unlocks *exactly one* consequential action. The runtime owns authority; the model only proposes intent.

Why the distinction matters. If approval is "a tool the model should call first," three failure modes are one prompt away:

1. the model **forgets** to call it and executes anyway;
2. the model gets approval for payload **A** and executes payload **B**;
3. the model **continues** after the approval path errored.

The fix: the executor refuses to dispatch the irreversible tool until there is an **approval record bound to the exact pending call**, and it **recomputes the binding from the pending call at execution time**. The model can be useful for intent and explanation; it is never the authority.

_(This executor-side framing follows review feedback on the cookbook proposal, gratefully adopted.)_


In [ ]:
# Pure-Python core — runs with no API key. The Claude agent loop comes at the end.
import json, hashlib, time

# The binding: a digest over the EXACT action — tool, version, canonical args,
# and the consequential fields (amount/recipient). Change any byte and it differs.
def action_digest(tool_name: str, tool_version: str, args: dict) -> str:
    canonical = json.dumps(
        {"tool": tool_name, "version": tool_version, "args": args},
        sort_keys=True, separators=(",", ":"),
    )
    return hashlib.sha256(canonical.encode()).hexdigest()


## 1. The approval record

An approval is a **record bound to one action digest**, carrying the approver's identity, an expiry, and a one-time-use flag. It is created by a human approving *that specific pending action* — never by the model asserting it happened.

Everything the executor needs to fail closed lives here: a missing record, a digest mismatch, a replay, an expiry, or a missing approver identity each terminates the action.


In [ ]:
SIMULATED_APPROVER_APPROVES = True  # flip to False to watch a proposal get refused

class ApprovalStore:
    """Executor-owned. In production each record is a device-bound human
    signature (WebAuthn/passkey) over the digest; here we simulate the human."""
    def __init__(self):
        self._records = {}  # digest -> record

    def request(self, digest: str, summary: str):
        # Route to a named human, block for their decision on THIS exact action.
        if not SIMULATED_APPROVER_APPROVES:
            return None
        self._records[digest] = {
            "digest": digest, "approver": "ops:jchen",
            "expires_at": time.time() + 300, "consumed": False, "summary": summary,
        }
        return self._records[digest]

    def consume_if_valid(self, digest: str):
        """The single gate. Returns (ok, info). Fails closed on every error path."""
        rec = self._records.get(digest)
        if rec is None:                      return (False, "no_matching_approval")
        if rec["consumed"]:                  return (False, "replay")
        if time.time() > rec["expires_at"]:  return (False, "expired")
        if not rec.get("approver"):          return (False, "missing_approver")
        rec["consumed"] = True               # one-time: consumed exactly once
        return (True, rec)


## 2. The executor owns authority

This is the heart of the pattern. When a tool call is irreversible, the executor **recomputes the digest from the pending call** and consumes a matching approval *before dispatch*. No record, no execution. The result links back to the approval, so a later reviewer can answer both questions: *what was approved* and *what actually executed*.


In [ ]:
IRREVERSIBLE = {"release_payment"}  # promote any high-risk tool into this set

def _dispatch(tool_name, args):  # the REAL side effect (stubbed)
    return {"status": "sent", **args}

def execute_tool(tool_name: str, args: dict, store: ApprovalStore) -> dict:
    if tool_name not in IRREVERSIBLE:
        return {"executed": True, "result": _dispatch(tool_name, args)}  # reversible: just run

    # Executor-side precondition. Recompute the binding from the pending call —
    # we do NOT trust any earlier 'approval' the model claims to have obtained.
    digest = action_digest(tool_name, "v1", args)
    ok, info = store.consume_if_valid(digest)
    if not ok:
        return {"executed": False, "reason": info, "approval_digest": digest}  # FAIL CLOSED

    result = _dispatch(tool_name, args)
    return {"executed": True, "result": result,
            "approval_digest": digest, "approver": info["approver"]}


## 3. A valid approval unlocks exactly one execution


In [ ]:
store = ApprovalStore()
action = {"wire_id": "wire/8841", "amount": 82000, "recipient": "Vendor 8841"}

# A human approves THIS pending action — the record is bound to its digest.
d = action_digest("release_payment", "v1", action)
store.request(d, summary="release $82,000 to Vendor 8841")

print("first execution: ", execute_tool("release_payment", action, store))
print("replay (same action):", execute_tool("release_payment", action, store))  # one-time → blocked


## 4. The "approved A, executes B" attack — caught

The most important failure mode to teach. A human approved **$82,000**. The executor is then asked to run **$820,000** (a tampered arg, a confused model, or a malicious intermediate). Because the executor recomputes the digest from the *pending* call, it never matches the approved record — so it fails closed. The approval was for one action, not a blank check.


In [ ]:
store2 = ApprovalStore()
approved = {"wire_id": "wire/8841", "amount": 82000,  "recipient": "Vendor 8841"}
tampered = {"wire_id": "wire/8841", "amount": 820000, "recipient": "Vendor 8841"}

store2.request(action_digest("release_payment", "v1", approved), "release $82,000")

print("approved $82k →", execute_tool("release_payment", approved, store2))
print("execute $820k →", execute_tool("release_payment", tampered, store2))  # no_matching_approval


## 5. Wiring it to a Claude agent loop

Now the model proposes; the executor enforces. The model may *plan* and *explain*; every irreversible `tool_use` is routed through `execute_tool`, which owns the approval gate. The model is never asked to gate itself — so it cannot forget to, or gate the wrong payload. (Needs `ANTHROPIC_API_KEY`.)


In [ ]:
# %pip install anthropic
import os, anthropic

RELEASE_PAYMENT_TOOL = {
    "name": "release_payment",
    "description": "Release a vendor payment (IRREVERSIBLE).",
    "input_schema": {"type": "object", "properties": {
        "wire_id": {"type": "string"}, "amount": {"type": "number"},
        "recipient": {"type": "string"}}, "required": ["wire_id", "amount", "recipient"]}}

def run_agent(prompt: str, store: ApprovalStore, max_turns: int = 6) -> list:
    client = anthropic.Anthropic()
    messages = [{"role": "user", "content": prompt}]
    outcomes = []
    for _ in range(max_turns):
        resp = client.messages.create(model="claude-fable-5", max_tokens=1024,
                                      tools=[RELEASE_PAYMENT_TOOL], messages=messages)
        if resp.stop_reason != "tool_use":
            return outcomes or ["".join(b.text for b in resp.content if b.type == "text")]
        messages.append({"role": "assistant", "content": resp.content})
        results = []
        for b in resp.content:
            if b.type != "tool_use":
                continue
            # The EXECUTOR decides — recompute, gate, dispatch-or-refuse.
            outcome = execute_tool(b.name, b.input, store)
            outcomes.append(outcome)
            results.append({"type": "tool_result", "tool_use_id": b.id,
                            "content": json.dumps(outcome)})
        messages.append({"role": "user", "content": results})
    return outcomes

if os.environ.get("ANTHROPIC_API_KEY"):
    store3 = ApprovalStore()
    # Pre-approve the exact action a human authorized out-of-band:
    store3.request(action_digest("release_payment", "v1",
        {"wire_id": "wire/8841", "amount": 82000, "recipient": "Vendor 8841"}),
        "release $82,000")
    print(run_agent("Release wire/8841 for $82,000 to Vendor 8841.", store3))
else:
    print("Set ANTHROPIC_API_KEY to run the live agent loop; the gate logic above runs without it.")


## What production needs beyond the simulation

The `SIMULATED_APPROVER_APPROVES` flag stands in for the one thing that makes the record trustworthy: **a named human's signature over the digest, produced on their own device** (WebAuthn / passkey), so neither a compromised agent nor the operator can forge an approval. Three properties to add:

1. **The record is a device-bound signature**, not a boolean — the approver's key never leaves their device.
2. **It is offline-verifiable** — any reviewer recomputes the digest and checks the signature, with no trust in the runtime.
3. **Separation of duties** — the approver is not the initiator, checked in protocol.

### Production references
- Vendor-neutral building blocks: [@simplewebauthn](https://simplewebauthn.dev/) (JS), [python-fido2](https://github.com/Yubico/python-fido2).
- [EMILIA Protocol](https://github.com/emiliaprotocol/emilia-protocol) (Apache-2.0) implements exactly this executor-side precondition as a tool-call gate — device-bound approver signatures over the canonical action digest, offline verification, separation of duties, one-time consumption — as a Claude Code plugin / Agent SDK hook and a [remote MCP connector](https://www.emiliaprotocol.ai/verify). Receipt format: [draft-schrock-ep-authorization-receipts](https://datatracker.ietf.org/doc/draft-schrock-ep-authorization-receipts/) (IETF).
- The strictness threshold maps to the *earned-trust* curve in [Anthropic's autonomy research](https://www.anthropic.com/news/measuring-agent-autonomy): gate everything at first, relax per-action as a clean approval history accrues.
